# 📈 Exercícios — Processamento Estatístico de Linguagem Natural

**Disciplina:** Inteligência Artificial | **Nível:** Avançado

> Explore N-gramas, modelos de linguagem, análise de sentimento e word embeddings simplificados.


## 1. Modelos de N-Gramas

In [ ]:
from collections import defaultdict, Counter
import re, math, random

def tokenizar(texto):
    return re.findall(r'\b\w+\b', texto.lower())

corpus = """
o gato comeu o rato o rato fugiu do gato o gato dorme
o cachorro late o cachorro come o cachorro brinca
o gato e o cachorro sao animais o gato dorme muito
o rato comeu o queijo o queijo sumiu do prato
"""

tokens = tokenizar(corpus)
print(f"Tokens: {len(tokens)}  |  Vocabulário: {len(set(tokens))}")

# Modelo de Bigramas
def construir_bigrama(tokens):
    modelo = defaultdict(Counter)
    for i in range(len(tokens)-1):
        modelo[tokens[i]][tokens[i+1]] += 1
    return modelo

def probabilidade_bigrama(modelo, w1, w2):
    total = sum(modelo[w1].values())
    return modelo[w1][w2] / total if total > 0 else 0

modelo_bigrama = construir_bigrama(tokens)
print("\nBigramas após 'o':", dict(modelo_bigrama['o'].most_common(5)))
print("\nProbabilidades condicionais após 'o':")
for w, c in modelo_bigrama['o'].most_common(5):
    p = probabilidade_bigrama(modelo_bigrama, 'o', w)
    print(f"  P({w}|o) = {p:.3f}  ({c} ocorrências)")


## 2. Geração de Texto com N-Gramas

In [ ]:
def gerar_texto_bigrama(modelo, inicio, n_palavras=15):
    """Gera texto usando o modelo de bigramas."""
    texto = [inicio]
    atual = inicio
    for _ in range(n_palavras-1):
        opcoes = modelo.get(atual)
        if not opcoes: break
        palavras = list(opcoes.keys())
        pesos = list(opcoes.values())
        total = sum(pesos)
        probs = [p/total for p in pesos]
        prox = random.choices(palavras, weights=probs)[0]
        texto.append(prox)
        atual = prox
    return ' '.join(texto)

random.seed(42)
print("Textos gerados pelo modelo de bigramas:")
for inicio in ['o', 'o gato', 'o cachorro']:
    palavra_inicio = inicio.split()[-1]
    texto = gerar_texto_bigrama(modelo_bigrama, palavra_inicio, n_palavras=12)
    print(f"  [{inicio}]: {texto}")


### 📝 Exercício 1

Implemente um modelo de **trigramas** e gere texto a partir dele. Compare a coerência com o modelo de bigramas.

In [ ]:
def construir_trigrama(tokens):
    """Modelo de trigramas: P(w3 | w1, w2)"""
    modelo = defaultdict(Counter)
    for i in range(len(tokens)-2):
        chave = (tokens[i], tokens[i+1])
        modelo[chave][tokens[i+2]] += 1
    return modelo

def gerar_texto_trigrama(modelo, w1, w2, n_palavras=15):
    texto = [w1, w2]
    for _ in range(n_palavras-2):
        opcoes = modelo.get((texto[-2], texto[-1]))
        if not opcoes: break
        palavras = list(opcoes.keys()); pesos = list(opcoes.values())
        prox = random.choices(palavras, weights=pesos)[0]
        texto.append(prox)
    return ' '.join(texto)

modelo_tri = construir_trigrama(tokens)
random.seed(42)
print("Trigramas gerados:")
print(" ", gerar_texto_trigrama(modelo_tri, 'o', 'gato'))
print(" ", gerar_texto_trigrama(modelo_tri, 'o', 'cachorro'))


## 3. Análise de Sentimento com Léxico

In [ ]:
# Léxico de sentimento simplificado
lexico = {
    "bom":1,"ótimo":2,"excelente":2,"maravilhoso":2,"gosto":1,"feliz":1,
    "adorei":2,"perfeito":2,"recomendo":1,"incrível":2,"lindo":1,
    "ruim":-1,"péssimo":-2,"terrível":-2,"horrível":-2,"odeio":-2,
    "detestei":-2,"decepcionante":-1,"fraco":-1,"triste":-1,"chato":-1,
    "não":-1,"nunca":-1,"jamais":-1,
}

def analisar_sentimento(texto):
    tokens = tokenizar(texto)
    score = 0
    negador = False
    for t in tokens:
        if t in ["não","nunca","jamais"]:
            negador = True
        elif t in lexico:
            s = lexico[t] * (-1 if negador else 1)
            score += s
            negador = False
        else:
            negador = False
    if score > 0: sentimento = "😊 Positivo"
    elif score < 0: sentimento = "😠 Negativo"
    else: sentimento = "😐 Neutro"
    return score, sentimento

avaliacoes = [
    "O produto é ótimo e maravilhoso, adorei!",
    "Terrível e péssimo, nunca mais compro",
    "Não gostei, muito fraco e decepcionante",
    "Excelente, perfeito, recomendo a todos",
    "Produto chegou no prazo",
    "Não é ruim mas também não é bom",
]
print(f"{'Avaliação':<50} | {'Score':^6} | Sentimento")
print("-"*75)
for av in avaliacoes:
    score, sent = analisar_sentimento(av)
    print(f"{av:<50} | {score:^6} | {sent}")


### 📝 Exercício 2

Adicione **10 novas palavras** ao léxico (5 positivas, 5 negativas) e crie 3 frases de teste para validar. Inclua pelo menos uma frase com negação complexa como *"não é totalmente ruim"*.

In [ ]:
lexico_expandido = dict(lexico)
# ✏️ Adicione novas palavras:
lexico_expandido.update({
    # positivas
    "satisfeito": 1, "eficiente": 1,
    # negativas
    "lento": -1, "defeituoso": -2,
    # TODO: adicione mais
})

minhas_frases = [
    "O atendimento foi eficiente e satisfatório",
    "O produto é lento e defeituoso",
    "não é totalmente ruim",  # negação complexa
]
for f in minhas_frases:
    score, sent = analisar_sentimento(f)
    print(f"'{f}' → {score} | {sent}")


## 4. Word Embeddings Simplificados — Co-ocorrência

In [ ]:
import numpy as np

def construir_matriz_coocorrencia(tokens, janela=2):
    vocab = sorted(set(tokens))
    v2i = {w:i for i,w in enumerate(vocab)}
    n = len(vocab)
    matriz = np.zeros((n,n))
    for i, w in enumerate(tokens):
        for j in range(max(0,i-janela), min(len(tokens),i+janela+1)):
            if i!=j: matriz[v2i[w], v2i[tokens[j]]] += 1
    return matriz, vocab

matriz_coo, vocab_coo = construir_matriz_coocorrencia(tokens, janela=2)
v2i = {w:i for i,w in enumerate(vocab_coo)}

# SVD para reduzir dimensionalidade → "embeddings"
U, S, Vt = np.linalg.svd(matriz_coo)
embeddings = U[:, :3] * np.sqrt(S[:3])  # top-3 componentes

def similaridade_embedding(w1, w2):
    if w1 not in v2i or w2 not in v2i: return 0
    e1, e2 = embeddings[v2i[w1]], embeddings[v2i[w2]]
    return float(np.dot(e1,e2)/(np.linalg.norm(e1)*np.linalg.norm(e2)+1e-10))

palavras_teste = [("gato","rato"),("gato","cachorro"),("gato","queijo"),("cachorro","rato")]
print("Similaridade por embeddings de co-ocorrência:")
for w1,w2 in palavras_teste:
    print(f"  sim({w1:10s}, {w2:10s}) = {similaridade_embedding(w1,w2):.4f}")


### 📝 Exercício Final

Usando a matriz de co-ocorrência e SVD, visualize os embeddings em 2D (use os 2 primeiros componentes). Palavras semântica
mente similares devem aparecer próximas no espaço.

In [ ]:
import matplotlib.pyplot as plt

# ✏️ Visualize os embeddings 2D
emb_2d = U[:, :2] * np.sqrt(S[:2])

plt.figure(figsize=(10,8))
for i, palavra in enumerate(vocab_coo):
    plt.scatter(emb_2d[i,0], emb_2d[i,1], s=50)
    plt.annotate(palavra, (emb_2d[i,0]+0.01, emb_2d[i,1]+0.01), fontsize=9)
plt.title('Word Embeddings 2D (Co-ocorrência + SVD)')
plt.xlabel('Componente 1'); plt.ylabel('Componente 2')
plt.grid(True); plt.show()
